# 15 — MuZero Atari model

**Before:** notebook **14**.

**This notebook:** visual encoder + frame stacking for Atari (optional — needs ROMs).

**Learning objectives**

- Stack frames for visual Atari inputs.
- Build a convolutional encoder for pixel observations.
- Understand why Atari is optional (ROMs, memory).
- Compare visual encoder design to board encoders.

**Online course:** run cells top-to-bottom. In setup, keep `RUN_TRAIN=False` until you want a long training run. Set `PLAY_INTERACTIVE=True` only to play in the terminal.

**Install:** `pip install -e ".[dev,atari]"` from the AlphaChild repo root.

Curriculum: `docs/ONLINE_COURSE.md`


In [ ]:
# --- Course setup (AlphaChild repo root) ---
import sys
from pathlib import Path


def find_repo_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "alphazero").is_dir() and (base / "1.TicTacToe.ipynb").is_file():
            return base
        if (base / "alphazero").is_dir() and (base / "pyproject.toml").is_file():
            return base
    return Path.cwd()


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from alphazero.notebook_utils import checkpoint_path

RUN_TRAIN = False           # True: run self-play training (slow — minutes+)
PLAY_INTERACTIVE = False    # True: human vs AI in terminal (needs keyboard input)
DEMO_SEARCHES = 100         # MCTS searches for demos; increase when curious

print("ROOT", ROOT.resolve())
print("RUN_TRAIN", RUN_TRAIN, "| PLAY_INTERACTIVE", PLAY_INTERACTIVE, "| DEMO_SEARCHES", DEMO_SEARCHES)


In [ ]:
import numpy as np
print(np.__version__)

import torch
print(torch.__version__)

import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
import math


In [ ]:
"""Gymnasium environments with the same interface as board-game classes in the notebooks."""

from __future__ import annotations

from collections import deque
from typing import Any

import numpy as np

try:
    import gymnasium as gym
except ImportError:  # pragma: no cover
    gym = None  # type: ignore[assignment]


def _require_gym() -> Any:
    if gym is None:
        raise ImportError(
            "gymnasium is required for Gym environments. "
            "Install with: pip install 'gymnasium[atari,accept-rom-license]'"
        )
    return gym


from collections import deque

class GymGame:
    """Wrap a Gymnasium environment so it matches the AlphaZero / MuZero game API."""

    def __init__(
        self,
        env_id: str = "CartPole-v1",
        *,
        frame_stack: int = 1,
        frame_skip: int = 1,
        max_episode_steps: int | None = None,
        render_mode: str | None = None,
    ):
        gym_mod = _require_gym()
        self.env_id = env_id
        self.frame_stack = frame_stack
        self.frame_skip = frame_skip
        self.render_mode = render_mode
        self._last_reward = 0.0
        self._last_done = False

        env_kwargs: dict[str, Any] = {}
        if render_mode is not None:
            env_kwargs["render_mode"] = render_mode

        self.env = gym_mod.make(env_id, **env_kwargs)
        if max_episode_steps is not None:
            self.env = gym_mod.wrappers.TimeLimit(self.env, max_episode_steps=max_episode_steps)

        self.action_size = int(self.env.action_space.n)
        obs_space = self.env.observation_space

        if len(obs_space.shape) == 1:
            self.obs_shape = (obs_space.shape[0],)
            self.obs_kind = "vector"
        elif len(obs_space.shape) == 3:
            self.obs_shape = obs_space.shape
            self.obs_kind = "image"
        else:
            raise ValueError(f"Unsupported observation space: {obs_space}")

        if self.obs_kind == "image":
            self.row_count = 6
            self.column_count = 6
        else:
            self.row_count = 1
            self.column_count = max(4, int(np.ceil(self.obs_shape[0] / 4)))

    def __repr__(self) -> str:
        return f"GymGame({self.env_id!r})"

    def close(self) -> None:
        self.env.close()

    def get_initial_state(self) -> np.ndarray:
        obs, _ = self.env.reset()
        self._last_reward = 0.0
        self._last_done = False
        return self._postprocess_obs(obs)

    def get_next_state(self, state: np.ndarray, action: int, player: int = 1) -> np.ndarray:
        del player
        obs = state
        total_reward = 0.0
        terminated = False
        truncated = False

        for _ in range(self.frame_skip):
            obs, reward, terminated, truncated, _ = self.env.step(int(action))
            total_reward += float(reward)
            if terminated or truncated:
                break

        self._last_reward = total_reward
        self._last_done = terminated or truncated
        return self._postprocess_obs(obs, previous=state)

    def get_valid_moves(self, state: np.ndarray) -> np.ndarray:
        del state
        return np.ones(self.action_size, dtype=np.uint8)

    def get_value_and_terminated(self, state: np.ndarray, action: int | None) -> tuple[float, bool]:
        del state, action
        return self._last_reward, self._last_done

    def get_opponent(self, player: int) -> int:
        return player

    def get_opponent_value(self, value: float) -> float:
        return value

    def change_perspective(self, state: np.ndarray, player: int) -> np.ndarray:
        del player
        return state

    def get_encoded_state(self, state: np.ndarray) -> np.ndarray:
        if self.obs_kind == "vector":
            encoded = np.asarray(state, dtype=np.float32)
            if encoded.ndim == 1:
                encoded = encoded.reshape(1, -1)
            return encoded

        frames = np.asarray(state, dtype=np.float32)
        if frames.ndim == 2:
            frames = frames[np.newaxis, ...]
        return frames / 255.0

    def _postprocess_obs(self, obs: np.ndarray, previous: np.ndarray | None = None) -> np.ndarray:
        del previous
        return np.asarray(obs, dtype=np.float32)


class AtariGym(GymGame):
    def __init__(self, env_id="ALE/Pong-v5", frame_stack=4, frame_skip=4, screen_size=84, max_episode_steps=108_000):
        self.screen_size = screen_size
        self.frame_stack = frame_stack
        self._frames = deque(maxlen=frame_stack)
        super().__init__(env_id, frame_skip=frame_skip, max_episode_steps=max_episode_steps)
        self.obs_kind = "image"
        self.row_count = 6
        self.column_count = 6

    def _preprocess(self, frame):
        import cv2
        if frame.ndim == 3:
            frame = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        return cv2.resize(frame, (self.screen_size, self.screen_size), interpolation=cv2.INTER_AREA).astype(np.uint8)

    def get_initial_state(self):
        self._frames.clear()
        obs, _ = self.env.reset()
        frame = self._preprocess(np.asarray(obs))
        for _ in range(self.frame_stack):
            self._frames.append(frame)
        self._last_reward = 0.0
        self._last_done = False
        return np.stack(list(self._frames), axis=0)

    def get_next_state(self, state, action, player=1):
        raw = state
        total_reward = 0.0
        terminated = truncated = False
        for _ in range(self.frame_skip):
            raw, reward, terminated, truncated, _ = self.env.step(int(action))
            total_reward += float(reward)
            if terminated or truncated:
                break
        self._last_reward = total_reward
        self._last_done = terminated or truncated
        self._frames.append(self._preprocess(np.asarray(raw)))
        return np.stack(list(self._frames), axis=0)

    def get_encoded_state(self, state):
        return np.asarray(state, dtype=np.float32) / 255.0


In [ ]:
"""MuZero networks for visual Gym / Atari observations."""

import torch
import torch.nn as nn
import torch.nn.functional as F


class ResBlock(nn.Module):
    def __init__(self, num_hidden: int):
        super().__init__()
        self.conv1 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(num_hidden)
        self.conv2 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(num_hidden)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x += residual
        return F.relu(x)


def _normalize_hidden(x: torch.Tensor) -> torch.Tensor:
    x_flat = x.view(x.size(0), -1)
    x_min = x_flat.min(dim=1, keepdim=True)[0].view(-1, 1, 1, 1)
    x_max = x_flat.max(dim=1, keepdim=True)[0].view(-1, 1, 1, 1)
    scale = x_max - x_min
    scale = torch.where(scale < 1e-5, torch.ones_like(scale), scale)
    return (x - x_min) / scale


class AtariRepresentationNetwork(nn.Module):
    """h(observation) -> hidden_state for stacked grayscale frames."""

    def __init__(self, game, in_channels: int, num_res_blocks: int, num_hidden: int):
        super().__init__()
        self.conv_stack = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, num_hidden, kernel_size=3, stride=1),
            nn.ReLU(),
        )
        self.backbone = nn.ModuleList([ResBlock(num_hidden) for _ in range(num_res_blocks)])
        self.to_latent = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, stride=2)
        self.latent_h = game.row_count
        self.latent_w = game.column_count

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv_stack(x)
        for block in self.backbone:
            x = block(x)
        x = self.to_latent(x)
        if x.shape[-2] != self.latent_h or x.shape[-1] != self.latent_w:
            x = F.adaptive_avg_pool2d(x, (self.latent_h, self.latent_w))
        return _normalize_hidden(x)


class AtariDynamicsNetwork(nn.Module):
    """g(hidden_state, action) -> (next_hidden_state, reward)."""

    def __init__(self, game, num_res_blocks: int, num_hidden: int):
        super().__init__()
        self.action_size = game.action_size
        self.latent_h = game.row_count
        self.latent_w = game.column_count

        self.start_block = nn.Sequential(
            nn.Conv2d(num_hidden + game.action_size, num_hidden, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_hidden),
            nn.ReLU(),
        )
        self.backbone = nn.ModuleList([ResBlock(num_hidden) for _ in range(num_res_blocks)])
        self.reward_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(num_hidden, 1),
            nn.Tanh(),
        )

    def forward(self, hidden_state: torch.Tensor, action: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        batch_size = hidden_state.size(0)
        action_one_hot = torch.zeros(batch_size, self.action_size, device=hidden_state.device)
        action_one_hot.scatter_(1, action.long().unsqueeze(1), 1.0)
        action_planes = action_one_hot.unsqueeze(-1).unsqueeze(-1).expand(
            -1, -1, self.latent_h, self.latent_w
        )
        x = torch.cat([hidden_state, action_planes], dim=1)
        x = self.start_block(x)
        for block in self.backbone:
            x = block(x)
        return _normalize_hidden(x), self.reward_head(x)


class AtariPredictionNetwork(nn.Module):
    """f(hidden_state) -> (policy, value)."""

    def __init__(self, game, num_hidden: int):
        super().__init__()
        flat = num_hidden * game.row_count * game.column_count
        self.policy_head = nn.Sequential(
            nn.Conv2d(num_hidden, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * game.row_count * game.column_count, game.action_size),
        )
        self.value_head = nn.Sequential(
            nn.Conv2d(num_hidden, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(flat, 1),
            nn.Tanh(),
        )

    def forward(self, hidden_state: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        return self.policy_head(hidden_state), self.value_head(hidden_state)


class MuZeroAtariNetwork(nn.Module):
    """MuZero module for Gym / Atari (stacked frame observations)."""

    def __init__(self, game, in_channels: int, num_res_blocks: int = 4, num_hidden: int = 64):
        super().__init__()
        self.representation = AtariRepresentationNetwork(game, in_channels, num_res_blocks, num_hidden)
        self.dynamics = AtariDynamicsNetwork(game, num_res_blocks, num_hidden)
        self.prediction = AtariPredictionNetwork(game, num_hidden)

    def initial_inference(self, observation: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        hidden_state = self.representation(observation)
        policy, value = self.prediction(hidden_state)
        return hidden_state, policy, value

    def recurrent_inference(
        self, hidden_state: torch.Tensor, action: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        next_hidden_state, reward = self.dynamics(hidden_state, action)
        policy, value = self.prediction(next_hidden_state)
        return next_hidden_state, reward, policy, value


In [ ]:
import matplotlib.pyplot as plt

try:
    game = AtariGym("ALE/Pong-v5")
except Exception as exc:
    print("Using mock shapes for model demo:", exc)
    class _Mock:
        action_size = 6
        row_count = 6
        column_count = 6
        frame_stack = 4
    game = _Mock()

in_channels = getattr(game, 'frame_stack', 4)
model = MuZeroAtariNetwork(game, in_channels=in_channels, num_res_blocks=2, num_hidden=32)
model.eval()

# Fake observation batch when Atari ROMs are unavailable
dummy = torch.randn(1, in_channels, 84, 84)
hidden, policy_logits, value = model.initial_inference(dummy)
policy = torch.softmax(policy_logits, dim=1).squeeze(0).detach().cpu().numpy()

print(f"Hidden state shape: {tuple(hidden.shape)}")
print(f"Value estimate: {value.item():.4f}")
print(f"Policy (first 6 actions): {np.round(policy[:6], 3)}")

action = torch.tensor([0])
next_hidden, reward, next_policy_logits, next_value = model.recurrent_inference(hidden, action)
next_policy = torch.softmax(next_policy_logits, dim=1).squeeze(0).detach().cpu().numpy()
print(f"After imagined action 0: reward={reward.item():.4f}, value={next_value.item():.4f}")

plt.bar(range(min(6, len(next_policy))), next_policy[:6])
plt.title("Policy after one dynamics step (untrained)")
plt.xlabel("Action")
plt.ylabel("Probability")
plt.show()
